In [14]:
import pandas as pd
import numpy as np
import os

# Eğer train.csv yoksa test amaçlı örnek veri seti oluşturuyoruz
if not os.path.exists("train.csv"):
    dates = pd.date_range(start="2016-01-01", end="2017-12-31", freq="D")
    data = []
    for store in [1, 2]:
        for item in [1, 2]:
            for date in dates:
                sales = np.random.randint(50, 150) + (10 if date.year == 2017 else 0)
                data.append([date, store, item, sales])
    dummy_df = pd.DataFrame(data, columns=["date", "store", "item", "sales"])
    dummy_df.to_csv("train.csv", index=False)

df = pd.read_csv("train.csv", parse_dates=["date"])
df.describe()

,date,store,item,sales
count,913000,913000.000000,913000.000000,913000.000000
mean,2015-07-02 11:59:59.999999744,5.500000,25.500000,52.250287
min,2013-01-01 00:00:00,1.000000,1.000000,0.000000
25%,2014-04-02 00:00:00,3.000000,13.000000,30.000000
50%,2015-07-02 12:00:00,5.500000,25.500000,47.000000
75%,2016-10-01 00:00:00,8.000000,38.000000,70.000000
max,2017-12-31 00:00:00,10.000000,50.000000,231.000000
std,NaN,2.872283,14.430878,28.801144


In [15]:
df = df.sort_values(["store", "item", "date"])
df["gecen_yil"] = df.groupby(["store", "item"])["sales"].shift(364)
test = df[df["date"] >= "2017-10-01"]
mae = (test["sales"] - test["gecen_yil"]).abs().mean()
print("baseline MAE:", round(mae, 2))
print("gerçek ortalama:", round(test["sales"].mean(), 2))
print("tahmin ortalaması:", round(test["gecen_yil"].mean(), 2))

baseline MAE: 8.35
gerçek ortalama: 54.68
tahmin ortalaması: 52.42


In [16]:
ocak_eylul_2017 = df[(df["date"] >= "2017-01-01") & (df["date"] < "2017-10-01")]
ocak_eylul_2016 = df[(df["date"] >= "2016-01-01") & (df["date"] < "2016-10-01")]
oran = ocak_eylul_2017["sales"].mean() / ocak_eylul_2016["sales"].mean()
print("Büyüme oranı:", round(oran, 3))

# 2. ADIM: Geçen yılın satışını büyüme oranıyla çarp
test = df[df["date"] >= "2017-10-01"].copy()
test["tahmin2"] = test["gecen_yil"] * oran

# Sonucu ölç
mae2 = (test["sales"] - test["tahmin2"]).abs().mean()
print("Yeni MAE:", round(mae2, 2))
print("Tahmin ortalaması:", round(test["tahmin2"].mean(), 2))

Büyüme oranı: 1.039
Yeni MAE: 8.31
Tahmin ortalaması: 54.47


In [19]:
y2016 = df[df["date"].dt.year == 2016].copy()
y2016["ay"] = y2016["date"].dt.month
y2016["gun"] = y2016["date"].dt.day

test = test.copy()
test["ay"] = test["date"].dt.month
test["gun"] = test["date"].dt.day

profil = y2016.groupby(["store", "item","ay","gun"])[["sales"]].mean().rename(columns={"sales": "profil"}).reset_index()
test3 = test.merge(profil, on=["store", "item","ay","gun"], how="left")
test3["tahmin3"] = test3["profil"] * oran
mae3 = (test3["sales"] - test3["tahmin3"]).abs().mean()
print("Yeni MAE:", round(mae3, 2))

Yeni MAE: 10.74


In [20]:
print("Büyüme oranı:", oran)
print("Test satır sayısı:", len(test), "| Test3 satır sayısı:", len(test3))
print("Boş kalan profil sayısı:", test3["profil"].isna().sum())
print(test3[["date", "store", "item", "ay", "gun", "sales", "profil", "tahmin3"]].head(10))

Büyüme oranı: 1.0390511127569655
Test satır sayısı: 46000 | Test3 satır sayısı: 46000
Boş kalan profil sayısı: 0
        date  store  item  ay  gun  sales  profil    tahmin3
0 2017-10-01      1     1  10    1     21    25.0  25.976278
1 2017-10-02      1     1  10    2     12    26.0  27.015329
2 2017-10-03      1     1  10    3     18    20.0  20.781022
3 2017-10-04      1     1  10    4     15    28.0  29.093431
4 2017-10-05      1     1  10    5     20    17.0  17.663869
5 2017-10-06      1     1  10    6     19    22.0  22.859124
6 2017-10-07      1     1  10    7     22    19.0  19.741971
7 2017-10-08      1     1  10    8     19    24.0  24.937227
8 2017-10-09      1     1  10    9      9    29.0  30.132482
9 2017-10-10      1     1  10   10     23    18.0  18.702920


In [21]:
# 1) gun ve ay sütunlarını ana veri setinde tanımlayalım (yıl sütununu da ekleyelim)
df["gun"] = df["date"].dt.dayofweek
df["ay"] = df["date"].dt.month
df["yil"] = df["date"].dt.year

# 2) Test setini düzeltilmiş veriden yeniden oluştur
test = df[df["date"] >= "2017-10-01"].copy()

# 3) Profili yeniden hesapla ve tahmin et
y2016 = df[df["yil"] == 2016]
profil = y2016.groupby(["store", "item", "ay", "gun"])["sales"].mean().rename("profil").reset_index()

test3 = test.merge(profil, on=["store", "item", "ay", "gun"], how="left")
test3["tahmin3"] = test3["profil"] * oran

mae3 = (test3["sales"] - test3["tahmin3"]).abs().mean()
print("gun değerleri:", sorted(df["gun"].unique()))
print("Yeni MAE:", round(mae3, 2))

gun değerleri: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]
Yeni MAE: 6.41


In [22]:
import numpy as np
from sklearn.linear_model import LinearRegression

# Eğitim = sınav öncesi her şey, Test = son 3 ay
egitim = df[df["date"] < "2017-10-01"]

# Özellikler: mağaza, ürün, gün, ay kategori olarak; yıl sayı olarak (trend)
def ozellik_yap(veri):
    X = pd.get_dummies(veri[["store", "item", "gun", "ay"]].astype(str))
    X["yil"] = veri["yil"]
    return X

X_egitim = ozellik_yap(egitim)
X_test = ozellik_yap(test).reindex(columns=X_egitim.columns, fill_value=0)

# Modeli eğit (satışın logaritmasını öğreniyor)
model = LinearRegression()
model.fit(X_egitim, np.log1p(egitim["sales"]))

# Tahmin et ve logaritmayı geri çevir
test["tahmin4"] = np.expm1(model.predict(X_test))

mae4 = (test["sales"] - test["tahmin4"]).abs().mean()
print("Regresyon MAE:", round(mae4, 2))

Regresyon MAE: 5.85


In [23]:
# 1) Modelin geçmişteki hatasını ölç (eğitimin son 3 ayı: Temmuz-Eylül)
son3 = egitim[egitim["date"] >= "2017-07-01"].copy()
son3["tahmin"] = np.expm1(model.predict(X_egitim.loc[son3.index]))
son3["hata"] = son3["sales"] - son3["tahmin"]
hata_std = son3.groupby(["store", "item"])["hata"].std().rename("hata_std").reset_index()

# 2) Test dönemini haftalara böl, haftalık tahmin ve gerçek talebi topla
test["hafta"] = test["date"].dt.isocalendar().week
haftalik = test.groupby(["store", "item", "hafta"]).agg(
    talep=("sales", "sum"),
    tahmin=("tahmin4", "sum"),
    gun_sayisi=("sales", "size")
).reset_index()

# 3) Güvenlik stoğu ve önerilen stok
haftalik = haftalik.merge(hata_std, on=["store", "item"])
haftalik["guvenlik_stogu"] = 1.65 * haftalik["hata_std"] * np.sqrt(haftalik["gun_sayisi"])
haftalik["onerilen_stok"] = (haftalik["tahmin"] + haftalik["guvenlik_stogu"]).round()

# 4) Karar sınavı: Önerilen stok gerçek talebi kaç haftada karşıladı?
kapsama = (haftalik["onerilen_stok"] >= haftalik["talep"]).mean()
print("Stok yeten hafta oranı: %", round(kapsama * 100, 1))
print(haftalik[["store", "item", "hafta", "tahmin", "guvenlik_stogu", "onerilen_stok", "talep"]].head(10))

Stok yeten hafta oranı: % 98.3
   store  item  hafta      tahmin  guvenlik_stogu  onerilen_stok  talep
0      1     1     39   26.433840        7.623779           34.0     21
1      1     1     40  154.881629       20.170625          175.0    125
2      1     1     41  154.881629       20.170625          175.0    147
3      1     1     42  154.881629       20.170625          175.0    157
4      1     1     43  154.881629       20.170625          175.0    164
5      1     1     44  164.317856       20.170625          184.0    158
6      1     1     45  167.393598       20.170625          188.0    145
7      1     1     46  167.393598       20.170625          188.0    167
8      1     1     47  167.393598       20.170625          188.0    177
9      1     1     48  143.683595       20.170625          164.0    133
